In [1]:
# notebook to generate vcfs of full prc set for generating Sei predictions

In [1]:
# import packages
import pandas as pd

In [2]:
# open prcs
# gtex
gtex_prc = pd.read_csv('../raw_data/gtex.paired.mpra.prc.txt', sep = '\t')
# ukbb
ukbb_prc = pd.read_csv('../raw_data/traits.paired.mpra.prc.txt', sep = '\t')

In [17]:
# make a bed file of cosmic variants for intersecting with promoter and other BED files
chrom4bed = []
start4bed = []
end4bed = []
id4bed = []
for variant in ukbb_prc['variant']:
    # update chromosome and id lists
    chrom4bed.append(variant.split(':')[0])
    id4bed.append(variant)
    # get reference allele
    ref = variant.split(':')[-2]
    # get alternate allele
    alt = variant.split(':')[-1]
    # get variant position
    pos = int(variant.split(':')[1])
    # check the length of the variant for generating the start and stop intervals
    if len(ref) == 1 and len(alt) == 1: # SNPs
        start4bed.append(pos - 1)
        end4bed.append(pos)
    elif len(ref) < len(alt): # Insertions
        start4bed.append(pos)
        end4bed.append(pos)
    elif len(ref) > len(alt): # Deletions
        start4bed.append(pos)
        end4bed.append(pos + len(ref) - 1)
    
ukbb_prc_bed = pd.DataFrame({0 : chrom4bed,
                             1 : start4bed,
                             2 : end4bed,
                             3 : id4bed})
# save to disk for liftover
# ukbb_prc_bed.to_csv('../processed_data/ukbb_prc_with_indels_hg19_4_liftover.bed', sep = '\t', header = None, index = False)

In [19]:
# open hg38 lifted bed file
ukbb_prc_hg38 = pd.read_csv('../processed_data/ukbb_prc_with_indels_hg38_lifted.bed', sep = '\t', header = None)

In [20]:
ukbb_prc_hg38.head()

,0,1,2,3,4
0,chr4,145835573,145835574,chr4:146756726:C:T,1
1,chr4,152049653,152049654,chr4:152970806:C:T,1
2,chr6,15375332,15375333,chr6:15375564:G:A,1
3,chr3,105804065,105804066,chr3:105522910:T:C,1
4,chr6,159069915,159069915,chr6:159490947:A:AT,1


In [31]:
# make vcfs for generating SEI predictions for comparison
gtex_prc_vcf = pd.DataFrame({'#CHROM' : [i.split('_')[0] for i in gtex_prc['variant_hg38']],
                            'POS' : [i.split('_')[1] for i in gtex_prc['variant_hg38']],
                            'ID' : ['.' for i in range(len(gtex_prc))],
                            'REF' : [i.split('_')[2] for i in gtex_prc['variant_hg38']],
                            'ALT' : [i.split('_')[3] for i in gtex_prc['variant_hg38']],
                            'QUALITY' : ['.' for i in range(len(gtex_prc))],
                            'FILTER' : ['.' for i in range(len(gtex_prc))],
                            'INFO' : gtex_prc['causal']})
traits_prc_vcf = pd.DataFrame({'#CHROM' : ukbb_prc_hg38[0],
                              'POS' : ukbb_prc_hg38[2],
                              'ID' : ukbb_prc_hg38[3],
                              'REF' : [i.split(':')[2] for i in ukbb_prc_hg38[3]],
                              'ALT' : [i.split(':')[3] for i in ukbb_prc_hg38[3]],
                              'QUALITY' : ['.' for i in range(len(ukbb_prc_hg38))],
                              'FILTER' : ['.' for i in range(len(ukbb_prc_hg38))],
                              'INFO' : ukbb_prc['causal']})
# save to disk
# gtex_prc_vcf.to_csv('../processed_data/gtex_prc_with_indels_hg38_vars_for_SEI_preds.vcf',
#                    sep = '\t',
#                    index = False)
# traits_prc_vcf.to_csv('../processed_data/traits_prc_with_indels_hg38_vars_for_SEI_preds.vcf',
#                      sep = '\t',
#                      index = False)

In [32]:
# run predictions with:
# sei_gtex_prc_with_indels.sh
# and
# sei_traits_prc_with_indels.sh
# at:
# /projects/tewhey-lab/buttsj/sei_model/sei-framework